In [1]:
# %pip install python-dotenv anthropic
import os, base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
client = Anthropic()
ASK = "In 8 words or fewer, what is this?"

IMAGES = {".png":"image/png", ".jpg":"image/jpeg", ".jpeg":"image/jpeg",
          ".gif":"image/gif", ".webp":"image/webp"}
TEXTS  = {".txt", ".md"}

def b64(raw):
    return base64.b64encode(raw).decode("utf-8")

def wrap(path):
    """Turn a file into (block_type, raw_bytes, content_block)."""
    ext = os.path.splitext(path)[1].lower()
    raw = open(path, "rb").read()
    if ext in IMAGES:
        return "image", raw, {"type":"image","source":{"type":"base64",
                "media_type":IMAGES[ext], "data":b64(raw)}}
    if ext == ".pdf":
        return "document", raw, {"type":"document","source":{"type":"base64",
                "media_type":"application/pdf", "data":b64(raw)}}
    return "text", raw, {"type":"text", "text":raw.decode("utf-8","replace")}

def status_of(e):
    code = getattr(e, "status_code", None)
    return str(code) if code else "error"

print("Key loaded from .env")

Key loaded from .env


In [ ]:
known = set(IMAGES) | TEXTS | {".pdf"}
files = sorted(f for f in os.listdir(".") if os.path.splitext(f)[1].lower() in known)

summary = []
for name in files:
    kind, raw, block = wrap(name)

    # OUT: what physically leaves the machine
    out = (f"{len(raw):,} B inline text" if kind == "text"
           else f"{len(raw):,} B -> {len(b64(raw)):,} b64 chars (+33%)")
    print(f"OUT  {name:22s} [{kind:8s} block]  {out}")

    # send a one-message frame
    try:
        r = client.messages.create(model="claude-haiku-4-5", max_tokens=20,
              messages=[{"role":"user","content":[block, {"type":"text","text":ASK}]}])
        reply = r.content[0].text.strip().replace("\n"," ")[:42]
        status, in_tok, out_tok = "200", r.usage.input_tokens, r.usage.output_tokens
    except Exception as e:
        status, in_tok, out_tok, reply = status_of(e), None, None, ""

    # IN: what comes back
    if status == "200":
        print(f"IN   <- 200   in={in_tok:>5} tok   out={out_tok:>3} tok   reply={reply!r}\n")
    else:
        print(f"IN   <- {status}  (rejected)\n")
    summary.append((name, kind, len(raw), status, in_tok))

In [ ]:
print(f"{'FILE':22s}{'TYPE':10s}{'OUT BYTES':>12s}{'STATUS':>9s}{'IN TOK':>9s}")
print("-" * 64)
for name, kind, nbytes, status, in_tok in summary:
    tok = f"{in_tok:,}" if in_tok is not None else "-"
    print(f"{name:22s}{kind:10s}{nbytes:>12,}{status:>9s}{tok:>9s}")
print("-" * 64)
print("Same frame every time - only the block type and the size change.")